# DENTEX SegAndDet Fast End-to-End Notebook
This notebook is configured for a fast experimental run with a target runtime near four hours on a single GPU.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/GalaxyShadesCat/dentex.git"
REPO_DIR = Path("/content/dentex")

try:
    import google.colab  # type: ignore
    BOOTSTRAP_IN_COLAB = True
except ImportError:
    BOOTSTRAP_IN_COLAB = False

if BOOTSTRAP_IN_COLAB:
    if REPO_DIR.exists():
        print("Repository exists, pulling latest changes...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    print("Working directory:", Path.cwd())
else:
    print("Not running in Colab. Skipping repository bootstrap.")


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

PROJECT_ROOT = Path.cwd()
CODE_ROOT = PROJECT_ROOT / 'DentexSegAndDet'
DATA_ROOT = PROJECT_ROOT / 'dentex_dataset'
CHECKPOINT_ROOT = CODE_ROOT / 'checkpoints'

print(PROJECT_ROOT)
print(CODE_ROOT)
print(DATA_ROOT)
print(CHECKPOINT_ROOT)
os.environ["DENTEX_SPLIT_SEED"] = "42"


In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = None
DENTEX_DRIVE_ROOT = None
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"IN_COLAB={IN_COLAB}")
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    if not DRIVE_ROOT.exists():
        raise FileNotFoundError("Google Drive mount point not found after mount.")
    DENTEX_DRIVE_ROOT = DRIVE_ROOT / "dentex"
    DENTEX_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    CACHE_ROOT = DENTEX_DRIVE_ROOT / "cache"
else:
    CACHE_ROOT = PROJECT_ROOT / ".cache"

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "hf_home")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface_hub")
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["ULTRALYTICS_HOME"] = str(CACHE_ROOT / "ultralytics")
os.environ["XDG_CACHE_HOME"] = str(CACHE_ROOT / "xdg")

if IN_COLAB:
    print(f"Drive mounted at: {DRIVE_ROOT}")
    print(f"Dentex Drive root: {DENTEX_DRIVE_ROOT}")
else:
    print("Not running in Google Colab. Skipping Drive mount.")

print("Cache root:", CACHE_ROOT)
print("HF_HOME:", os.environ["HF_HOME"])
print("TORCH_HOME:", os.environ["TORCH_HOME"])
print("ULTRALYTICS_HOME:", os.environ["ULTRALYTICS_HOME"])


In [ ]:
def ensure_symlink(link_path: Path, target_path: Path):
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        if link_path.resolve() != target_path.resolve():
            link_path.unlink()
            os.symlink(target_path.resolve(), link_path)
    elif link_path.exists():
        pass
    else:
        os.symlink(target_path.resolve(), link_path)

if IN_COLAB and DENTEX_DRIVE_ROOT is not None:
    ensure_symlink(CODE_ROOT / "checkpoints", DENTEX_DRIVE_ROOT / "checkpoints")
    ensure_symlink(CODE_ROOT / "results", DENTEX_DRIVE_ROOT / "results")
    ensure_symlink(CODE_ROOT / "runs", DENTEX_DRIVE_ROOT / "runs")
    for output_name in [
        "output_diffdet_quadrant_fast",
        "output_dino_res50_enum32_fast",
        "output_dino_swin_disease_fast",
        "output_unet_enum9_unet_fast",
        "output_unet_enum9_seunet_fast",
        "output_unet_enum32_unet_fast",
        "output_unet_enum32_seunet_fast",
    ]:
        ensure_symlink(CODE_ROOT / output_name, DENTEX_DRIVE_ROOT / output_name)
    print("Linked checkpoints/results/runs/outputs to Google Drive dentex directory.")


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "gdown", "requests"], check=True)


In [ ]:
from datetime import datetime

if IN_COLAB and DENTEX_DRIVE_ROOT is not None:
    LOG_ROOT = DENTEX_DRIVE_ROOT / "run_logs"
else:
    LOG_ROOT = PROJECT_ROOT / "run_logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=CODE_ROOT, env=None, log_name=None):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if log_name is None:
        log_name = Path(cmd[1]).stem if len(cmd) > 1 else "command"
    log_path = LOG_ROOT / f"{timestamp}_{log_name}.log"

    print(f"\n>>> Running: {' '.join(cmd)}")
    print(f">>> CWD: {cwd}")
    print(f">>> Logging to: {log_path}")

    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=cwd,
            env=merged_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

    print(f">>> Completed: {' '.join(cmd)}")

def report_paths(title, paths):
    print(f"\n=== {title} ===")
    for path in paths:
        exists = path.exists()
        size = path.stat().st_size if exists and path.is_file() else 0
        print(f"{path}: exists={exists} size={size}")


## 1) Dataset Availability

In [ ]:
from huggingface_hub import snapshot_download

if IN_COLAB and DRIVE_ROOT is not None:
    PERSIST_ROOT = DENTEX_DRIVE_ROOT
else:
    PERSIST_ROOT = PROJECT_ROOT
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {
    "quadrant": ["train_quadrant.json"],
    "quadrant_enumeration": ["train_quadrant_enumeration.json"],
    "quadrant_enumeration_disease": [
        "train_quadrant_enumeration_disease.json",
        "train_triple.json",
        "training_triple.json",
        "triple.json",
    ],
}

CANONICAL_NAMES = {
    "quadrant": "train_quadrant.json",
    "quadrant_enumeration": "train_quadrant_enumeration.json",
    "quadrant_enumeration_disease": "train_quadrant_enumeration_disease.json",
}

ARCHIVE_NAMES = ["training_data.zip", "validation_data.zip", "test_data.zip"]

def subset_ready(root: Path, subset_name: str) -> bool:
    subset_dir = root / "origin" / subset_name
    return any((subset_dir / name).exists() for name in EXPECTED_FILES[subset_name])

def is_dataset_ready(root: Path) -> bool:
    return all(subset_ready(root, subset_name) for subset_name in EXPECTED_FILES)

def archives_present(zip_root: Path) -> bool:
    return all((zip_root / archive_name).exists() for archive_name in ARCHIVE_NAMES)

def extract_required_archives(zip_root: Path, extract_root: Path):
    extract_root.mkdir(parents=True, exist_ok=True)
    for archive_name in ARCHIVE_NAMES:
        archive_path = zip_root / archive_name
        if archive_path.exists():
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(extract_root)

def find_dir_with_any_json(base_dir: Path, names: list[str]):
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0].parent, matches[0].name
    return None, None

def resolve_subset_dir(extract_root: Path, subset_name: str) -> tuple[Path, str]:
    expected_names = EXPECTED_FILES[subset_name]
    candidate_dirs = [
        extract_root / "training_data" / subset_name,
        extract_root / "training_data" / "origin" / subset_name,
        extract_root / subset_name,
        extract_root / "training_data",
    ]
    for candidate_dir in candidate_dirs:
        if not candidate_dir.exists():
            continue
        for json_name in expected_names:
            if (candidate_dir / json_name).exists():
                return candidate_dir, json_name

    recursive_dir, recursive_name = find_dir_with_any_json(extract_root, expected_names)
    if recursive_dir is not None:
        return recursive_dir, recursive_name

    raise FileNotFoundError(
        f"Could not locate any of {expected_names} for {subset_name} under {extract_root}"
    )

def mirror_dir(src_dir: Path, dst_dir: Path):
    if dst_dir.is_symlink() or dst_dir.is_file():
        dst_dir.unlink()
    elif dst_dir.exists():
        shutil.rmtree(dst_dir, ignore_errors=True)
    try:
        os.symlink(src_dir.resolve(), dst_dir)
    except OSError:
        if dst_dir.is_symlink() or dst_dir.is_file():
            dst_dir.unlink()
        elif dst_dir.exists():
            shutil.rmtree(dst_dir, ignore_errors=True)
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)

def ensure_canonical_json(dst_subset_dir: Path, subset_name: str, found_name: str):
    canonical = dst_subset_dir / CANONICAL_NAMES[subset_name]
    if canonical.exists():
        return
    src = dst_subset_dir / found_name
    if src.exists() and src != canonical:
        shutil.copy2(src, canonical)

def normalise_origin_layout(extract_root: Path, target_root: Path):
    origin_root = target_root / "origin"
    origin_root.mkdir(parents=True, exist_ok=True)

    for subset_name in ["quadrant", "quadrant_enumeration", "quadrant_enumeration_disease"]:
        src_subset_dir, found_name = resolve_subset_dir(extract_root, subset_name)
        dst_subset_dir = origin_root / subset_name
        mirror_dir(src_subset_dir, dst_subset_dir)
        ensure_canonical_json(dst_subset_dir, subset_name, found_name)

target_dataset_root = PERSIST_ROOT / "dentex_dataset"
downloads_root = PERSIST_ROOT / "downloads" / "DENTEX"
zip_root = downloads_root / "DENTEX"
extract_root = downloads_root / "extracted"
extract_marker = extract_root / ".extract_done"
normalise_marker = target_dataset_root / ".normalise_done"

if not is_dataset_ready(target_dataset_root):
    if not archives_present(zip_root):
        print("Downloading dataset snapshot...")
        snapshot_download(
            repo_id="ibrahimhamamci/DENTEX",
            repo_type="dataset",
            local_dir=str(downloads_root),
        )
    else:
        print(f"Archives already present at {zip_root}. Skipping download.")

    if not extract_marker.exists():
        print("Extracting archives...")
        extract_required_archives(zip_root, extract_root)
        extract_marker.parent.mkdir(parents=True, exist_ok=True)
        extract_marker.write_text("ok", encoding="utf-8")
    else:
        print(f"Extraction marker found at {extract_marker}. Skipping extraction.")

    if not normalise_marker.exists() or not is_dataset_ready(target_dataset_root):
        print("Normalising origin layout...")
        normalise_origin_layout(extract_root, target_dataset_root)
        normalise_marker.parent.mkdir(parents=True, exist_ok=True)
        normalise_marker.write_text("ok", encoding="utf-8")
    else:
        print(f"Normalisation marker found at {normalise_marker}. Skipping normalisation.")
else:
    print(f"Dataset already ready at {target_dataset_root}. Skipping download/extract/normalise.")

if not is_dataset_ready(target_dataset_root):
    raise FileNotFoundError("DENTEX dataset with expected origin structure was not found.")

DATA_ROOT = target_dataset_root.resolve()
print("Using dataset root:", DATA_ROOT)

target_link = CODE_ROOT / "dentex_dataset"
if target_link.is_symlink() and target_link.resolve() != DATA_ROOT:
    target_link.unlink()
if not target_link.exists():
    try:
        os.symlink(DATA_ROOT, target_link)
    except OSError:
        if not target_link.exists():
            shutil.copytree(DATA_ROOT, target_link, dirs_exist_ok=True)


## 2) Pretrained Files

In [ ]:
import gdown
from ultralytics import YOLO
from urllib.request import urlretrieve

if IN_COLAB and DRIVE_ROOT is not None:
    PERSIST_CHECKPOINT_ROOT = DENTEX_DRIVE_ROOT / "checkpoints"
else:
    PERSIST_CHECKPOINT_ROOT = CHECKPOINT_ROOT
PERSIST_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
def ensure_http_download(url: str, destination: Path):
    if destination.exists():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(url, destination)
def ensure_gdrive_download(file_id: str, destination: Path):
    if destination.exists():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=file_id, output=str(destination), quiet=False)
ensure_http_download(
    "https://github.com/ShoufaChen/DiffusionDet/releases/download/v0.1/swin_base_patch4_window7_224_22k.pkl",
    PERSIST_CHECKPOINT_ROOT / "swin_base_patch4_window7_224_22k.pkl",
)
ensure_gdrive_download("1AwUn5EebmmLBo7njjW_Ng1q9zDrqkNbB", PERSIST_CHECKPOINT_ROOT / "dino_pretrained_checkpoint0033_4scale.pth")
ensure_gdrive_download("1CrzFP0RycSC24KKmF5k0libLRJgpX9x0", PERSIST_CHECKPOINT_ROOT / "dino_pretrained_checkpoint0029_4scale_swin.pth")
yolo_model = YOLO("yolov8x.pt")
shutil.copy2(Path(yolo_model.ckpt_path), PERSIST_CHECKPOINT_ROOT / "yolov8x.pt")
for required_file in [
    "swin_base_patch4_window7_224_22k.pkl",
    "dino_pretrained_checkpoint0033_4scale.pth",
    "dino_pretrained_checkpoint0029_4scale_swin.pth",
    "yolov8x.pt",
]:
    src = PERSIST_CHECKPOINT_ROOT / required_file
    dst = CHECKPOINT_ROOT / required_file
    src_resolved = src.resolve()
    dst_resolved = dst.resolve() if dst.exists() else dst
    if src_resolved != dst_resolved:
        shutil.copy2(src, dst)
    print(required_file, dst.exists(), dst.stat().st_size if dst.exists() else 0)
print("Persistent checkpoint cache:", PERSIST_CHECKPOINT_ROOT)


## 3) Preprocessing

In [ ]:
sys.path.insert(0, str(CODE_ROOT))
os.chdir(CODE_ROOT)
import process_dataset

process_dataset.process_coco_quadrant()
process_dataset.process_coco_enumeration32()
process_dataset.process_coco_disease()
process_dataset.process_coco_disease_all()
process_dataset.process_yolo_disease_all()
process_dataset.process_yolo_disease()
process_dataset.process_seg_enumeration32()
process_dataset.process_seg_enumeration9('results/enumeration_dataset_quadrant_predictions.json')


## 4) Fast Profile Configuration

In [ ]:
FAST_PROFILE = True

# Consistent sample caps across tasks for predictable runtime.
TRAIN_CAP = 200
VAL_CAP = 40
SEG_CAP = 200
TEST_CAP = 40

SAMPLE_CAPS = {
    "quadrant_train": TRAIN_CAP,
    "quadrant_val": VAL_CAP,
    "enum32_train": TRAIN_CAP,
    "enum32_val": VAL_CAP,
    "disease_train": TRAIN_CAP,
    "disease_val": VAL_CAP,
    "disease_all_train": TRAIN_CAP,
    "disease_all_val": VAL_CAP,
    "seg_enum32": SEG_CAP,
    "seg_enum9": SEG_CAP,
    "yolo_train": TRAIN_CAP,
    "yolo_val": VAL_CAP,
}

DIFFDET_MAX_ITER = 600
DIFFDET_EVAL_PERIOD = 200
DINO_EPOCHS = 2
YOLO_EPOCHS = 8
UNET_EPOCHS_UNET = 8
UNET_EPOCHS_SEUNET = 12

os.environ["DENTEX_TEST_MAX_SLICES"] = str(TEST_CAP)
print("FAST_PROFILE:", FAST_PROFILE)


In [ ]:
def subset_coco_json(json_path: Path, max_images: int):
    data = json.loads(json_path.read_text())
    images = sorted(data["images"], key=lambda x: x["id"])[:max_images]
    image_ids = {img["id"] for img in images}
    annotations = [ann for ann in data["annotations"] if ann["image_id"] in image_ids]
    subset = {"images": images, "annotations": annotations, "categories": data["categories"]}
    json_path.write_text(json.dumps(subset))

def cap_segmentation_image_names(json_path: Path, max_images: int):
    names = json.loads(json_path.read_text())
    names = names[:max_images]
    json_path.write_text(json.dumps(names))

def build_fast_yolo_dataset(src_root: Path, dst_root: Path, max_train: int, max_val: int):
    if dst_root.exists():
        shutil.rmtree(dst_root)
    for split, limit in [("train2017", max_train), ("val2017", max_val)]:
        src_img = src_root / "images" / split
        src_lbl = src_root / "labels" / split
        dst_img = dst_root / "images" / split
        dst_lbl = dst_root / "labels" / split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        selected = sorted([p for p in src_img.iterdir() if p.is_file()])[:limit]
        for image_path in selected:
            label_path = src_lbl / f"{image_path.stem}.txt"
            if label_path.exists():
                shutil.copy2(image_path, dst_img / image_path.name)
                shutil.copy2(label_path, dst_lbl / label_path.name)

if FAST_PROFILE:
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/quadrant/annotations/instances_train2017.json", SAMPLE_CAPS["quadrant_train"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/quadrant/annotations/instances_val2017.json", SAMPLE_CAPS["quadrant_val"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/enumeration32/annotations/instances_train2017.json", SAMPLE_CAPS["enum32_train"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/enumeration32/annotations/instances_val2017.json", SAMPLE_CAPS["enum32_val"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/disease/annotations/instances_train2017.json", SAMPLE_CAPS["disease_train"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/disease/annotations/instances_val2017.json", SAMPLE_CAPS["disease_val"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/disease_all/annotations/instances_train2017.json", SAMPLE_CAPS["disease_all_train"])
    subset_coco_json(CODE_ROOT / "dentex_dataset/coco/disease_all/annotations/instances_val2017.json", SAMPLE_CAPS["disease_all_val"])

    cap_segmentation_image_names(CODE_ROOT / "dentex_dataset/segmentation/enumeration32/image_names.json", SAMPLE_CAPS["seg_enum32"])
    cap_segmentation_image_names(CODE_ROOT / "dentex_dataset/segmentation/enumeration9/image_names.json", SAMPLE_CAPS["seg_enum9"])

    build_fast_yolo_dataset(
        CODE_ROOT / "dentex_dataset/yolo/disease",
        CODE_ROOT / "dentex_dataset/yolo/disease_fast",
        SAMPLE_CAPS["yolo_train"],
        SAMPLE_CAPS["yolo_val"],
    )

    yolo_fast_yaml = CODE_ROOT / "configs/yolo/dentex_disease_dataset_fast.yaml"
    yolo_fast_yaml.write_text(
        "\n".join([
            "path: ../dentex_dataset/yolo/disease_fast",
            "train: images/train2017",
            "val: images/val2017",
            "names:",
            "  0: Impacted",
            "  1: Caries",
            "  2: Periapical Lesion",
            "  3: Deep Caries",
        ])
    )

print("Fast profile subsets prepared.")


## 5) Augmentation

In [ ]:
print("Augmentation is applied internally by train_unet.py.")


## 6) Training

In [ ]:
run_cmd([sys.executable, "setup.py", "install"], cwd=CODE_ROOT / "models" / "dino" / "ops", env={"TORCH_CUDA_ARCH_LIST": "7.5;8.6"})


In [ ]:
run_cmd([
    sys.executable, "train_diffdet.py",
    "--output-dir", "output_diffdet_quadrant_fast",
    "--config-file", "configs/diffdet/diffdet.dentex.swinbase.quadrant.yaml",
    "SOLVER.MAX_ITER", str(DIFFDET_MAX_ITER),
    "TEST.EVAL_PERIOD", str(DIFFDET_EVAL_PERIOD),
    "SOLVER.CHECKPOINT_PERIOD", str(DIFFDET_EVAL_PERIOD),
    "MODEL.WEIGHTS", "checkpoints/swin_base_patch4_window7_224_22k.pkl",
], log_name="train_diffdet")

report_paths("DiffDet Outputs", [
    CODE_ROOT / "output_diffdet_quadrant_fast" / "model_final.pth",
    CODE_ROOT / "output_diffdet_quadrant_fast" / "metrics.json",
])


In [ ]:
run_cmd([
    sys.executable, "train_dino.py",
    "--output_dir", "output_dino_res50_enum32_fast",
    "-c", "configs/dino/DINO_4scale_cls32.py",
    "--coco_path", "dentex_dataset/coco/enumeration32",
    "--num_workers", "2",
    "--options",
    "dn_scalar=100", "embed_init_tgt=TRUE", "dn_label_coef=1.0", "dn_bbox_coef=1.0", "use_ema=False", "dn_box_noise_scale=1.0",
    f"epochs={DINO_EPOCHS}", "lr_drop=1", "save_checkpoint_interval=1",
    "--pretrain_model_path", "checkpoints/dino_pretrained_checkpoint0033_4scale.pth",
    "--finetune_ignore", "label_enc.weight", "class_embed",
], log_name="train_dino_enum32")

run_cmd([
    sys.executable, "train_dino.py",
    "--output_dir", "output_dino_swin_disease_fast",
    "-c", "configs/dino/DINO_4scale_swin_cls4.py",
    "--coco_path", "dentex_dataset/coco/disease_all",
    "--num_workers", "2",
    "--options",
    "dn_scalar=100", "embed_init_tgt=TRUE", "dn_label_coef=1.0", "dn_bbox_coef=1.0", "use_ema=False", "dn_box_noise_scale=1.0",
    f"epochs={DINO_EPOCHS}", "lr_drop=1", "save_checkpoint_interval=1",
    "--pretrain_model_path", "checkpoints/dino_pretrained_checkpoint0029_4scale_swin.pth",
    "--finetune_ignore", "label_enc.weight", "class_embed",
], log_name="train_dino_disease")

report_paths("DINO Outputs", [
    CODE_ROOT / "output_dino_res50_enum32_fast" / "checkpoint.pth",
    CODE_ROOT / "output_dino_swin_disease_fast" / "checkpoint.pth",
])


In [ ]:
from ultralytics import YOLO

yolo = YOLO(str(CHECKPOINT_ROOT / "yolov8x.pt"), task="detect")
yolo.train(
    data=str(CODE_ROOT / "configs" / "yolo" / "dentex_disease_dataset_fast.yaml"),
    epochs=YOLO_EPOCHS,
    project=str(CODE_ROOT / "runs" / "detect"),
    name="dentex_fast",
    exist_ok=True,
    save_period=1,
    verbose=True,
)

report_paths("YOLO Outputs", [
    CODE_ROOT / "runs" / "detect" / "dentex_fast" / "weights" / "best.pt",
    CODE_ROOT / "runs" / "detect" / "dentex_fast" / "weights" / "last.pt",
])


In [ ]:
run_cmd([
    sys.executable, "train_unet.py",
    "--output_dir", "output_unet_enum9_unet_fast",
    "--dataset_dir", "dentex_dataset/segmentation/enumeration9",
    "--num_classes", "9",
    "--model", "unet",
    "--epochs", str(UNET_EPOCHS_UNET),
    "--batch_size", "16",
    "--save_interval", "2",
], log_name="train_unet_enum9_unet")

run_cmd([
    sys.executable, "train_unet.py",
    "--output_dir", "output_unet_enum9_seunet_fast",
    "--dataset_dir", "dentex_dataset/segmentation/enumeration9",
    "--num_classes", "9",
    "--model", "seunet",
    "--epochs", str(UNET_EPOCHS_SEUNET),
    "--batch_size", "16",
    "--save_interval", "2",
], log_name="train_unet_enum9_seunet")

run_cmd([
    sys.executable, "train_unet.py",
    "--output_dir", "output_unet_enum32_unet_fast",
    "--dataset_dir", "dentex_dataset/segmentation/enumeration32",
    "--num_classes", "32",
    "--model", "unet",
    "--epochs", str(UNET_EPOCHS_UNET),
    "--batch_size", "16",
    "--save_interval", "2",
], log_name="train_unet_enum32_unet")

run_cmd([
    sys.executable, "train_unet.py",
    "--output_dir", "output_unet_enum32_seunet_fast",
    "--dataset_dir", "dentex_dataset/segmentation/enumeration32",
    "--num_classes", "32",
    "--model", "seunet",
    "--epochs", str(UNET_EPOCHS_SEUNET),
    "--batch_size", "16",
    "--save_interval", "2",
], log_name="train_unet_enum32_seunet")

report_paths("UNet Outputs", [
    CODE_ROOT / "output_unet_enum9_unet_fast" / "auto_save.pth",
    CODE_ROOT / "output_unet_enum9_unet_fast" / "last_epoch.pth",
    CODE_ROOT / "output_unet_enum9_seunet_fast" / "auto_save.pth",
    CODE_ROOT / "output_unet_enum9_seunet_fast" / "last_epoch.pth",
    CODE_ROOT / "output_unet_enum32_unet_fast" / "auto_save.pth",
    CODE_ROOT / "output_unet_enum32_unet_fast" / "last_epoch.pth",
    CODE_ROOT / "output_unet_enum32_seunet_fast" / "auto_save.pth",
    CODE_ROOT / "output_unet_enum32_seunet_fast" / "last_epoch.pth",
])


## 7) Checkpoint Mapping and Testing

In [ ]:
def copy_ckpt(src: Path, dst_name: str):
    dst = CHECKPOINT_ROOT / dst_name
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, dst)
    print(src, "->", dst)

copy_ckpt(CODE_ROOT / "output_dino_res50_enum32_fast" / "checkpoint.pth", "dino_res50_enum_24.pth")
copy_ckpt(CODE_ROOT / "output_dino_swin_disease_fast" / "checkpoint.pth", "dino_swin_disease_all_27.pth")
copy_ckpt(CODE_ROOT / "runs" / "detect" / "dentex_fast" / "weights" / "best.pt", "yolo_disease_all_25.pt")
copy_ckpt(CODE_ROOT / "output_unet_enum9_unet_fast" / "last_epoch.pth", "unet_3_epoch53.pth")
copy_ckpt(CODE_ROOT / "output_unet_enum9_unet_fast" / "last_epoch.pth", "unet_3_epoch240.pth")
copy_ckpt(CODE_ROOT / "output_unet_enum9_seunet_fast" / "last_epoch.pth", "seunet_3_epoch15.pth")
copy_ckpt(CODE_ROOT / "output_unet_enum32_unet_fast" / "last_epoch.pth", "unet33_1_epoch175.pth")
copy_ckpt(CODE_ROOT / "output_unet_enum32_seunet_fast" / "last_epoch.pth", "seunet33_1_epoch132.pth")
copy_ckpt(CODE_ROOT / "output_unet_enum32_seunet_fast" / "last_epoch.pth", "seunet33_1_epoch250.pth")
copy_ckpt(CODE_ROOT / "output_diffdet_quadrant_fast" / "model_final.pth", "diffdet_quadrant.pth")


In [ ]:
print("Inference slice cap:", os.environ.get("DENTEX_TEST_MAX_SLICES", ""))
run_cmd([sys.executable, "predict.py"])
print((CODE_ROOT / "results" / "abnormal-teeth-detection.json").resolve())
if IN_COLAB and DENTEX_DRIVE_ROOT is not None:
    print("Drive dentex root:", DENTEX_DRIVE_ROOT)
    print("Drive results path:", DENTEX_DRIVE_ROOT / "results" / "abnormal-teeth-detection.json")
